## Evaluation — Combining Scores and Selecting a Detection Threshold

The two scoring layers built earlier(the global Mahalanobis baseline and the time-adaptive EWMA baseline) are combined here into a single score and used to select a concrete decision threshold. The used score is the maximum of the two scores per transaction: a transaction is treated as more anomalous if *either* baseline considers it unusual, since the adaptive layer exists specifically to catch anomalies the global baseline misses due to time-of-day drift, and vice versa.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
import bisect

path = '../data/findings.csv'

findings = pd.read_csv(path)
fraud = findings.loc[findings.Class == 1].to_numpy()
non_fraud = findings.loc[findings.Class == 0].to_numpy()

used_scores = np.array([max(row[2], row[3]) for row in fraud])
fmean, fmaxx, fminn, fstd = np.mean(used_scores), np.max(used_scores), np.min(used_scores), np.sqrt(np.var(used_scores, ddof=1))

reg_scores = np.array([max(row[2], row[3]) for row in non_fraud])
rmean, rmaxx, rminn, rstd = np.mean(reg_scores), np.max(reg_scores), np.min(reg_scores), np.sqrt(np.var(reg_scores, ddof=1))

print(f"[Standard] Mean: {rmean}, max: {rmaxx}, min: {rminn}, std: {rstd}")
print(f"[Fraud] Mean: {fmean}, max: {fmaxx}, min: {fminn}, std: {fstd}")



[Standard] Mean: 4.6822734979017, max: 152.03246329033195, min: 2.0605063142622644, std: 2.7929048217562
[Fraud] Mean: 57.22489745775598, max: 148.84610532341327, min: 3.272100227808851, std: 39.99186309368643


**Finding:** the two class-conditional score distributions overlap significantly. The lowest fraud score falls below the mean non-fraud score, and the single highest score in the entire dataset belongs to a non-fraud transaction, exceeding even the highest fraud score. This rules out any threshold achieving both perfect recall and zero false positives. It also doesn't make sense to use Gaussians to describe the two distributions since there would be sizeable probability mass below 0 (impossiblle for the scores, generated through Mahalanobis distance, to be negative).  The threshold is selected empirically below, by sweeping every possible cutoff against the true labels.

In [2]:
transactions = findings.to_numpy()
transactions = np.array([[row[0], row[1], max(row[2], row[3])] for row in transactions])
transactions = pd.DataFrame(data=transactions, columns=['Time', 'Class', 'Score']).sort_values(by='Score', ascending=False, ignore_index=True).to_numpy()

# FPR = FP / (FP + TN)
# Recall = TP / (TP + FN)
# Precision = TP / (TP + FP)
num_frauds = len(fraud)
num_reg = len(non_fraud)
# contains rows of [threshold score, FPR, Recall, Precision] where any score s >= threshold is marked as fraud
metrics = []
seen_frauds = 0
for index, score in enumerate(transactions[:, 2]):
    predicted_frauds = index + 1
    if transactions[index][1] == 1:
        seen_frauds += 1
    tp = seen_frauds
    fp = predicted_frauds - seen_frauds
    fn = num_frauds - seen_frauds
    tn = num_reg - fp
    fpr = fp / (fp + tn)
    recall = tp / (tp + fn)
    precision = tp/ (tp + fp)
    metrics.append([score, fpr, recall, precision])

### Building the full threshold sweep

Every transaction's combined score is treated as a candidate cutoff, with scores $\ge$ the threshold being marked as fraud. Sorting all transactions by score in descending order and walking down the list one row at a time reproduces the entire precision/recall/false-positive-rate curve in a single pass: visiting a given row is equivalent to lowering the threshold to that row's exact score, and a running count of true and false positives at that point in the walk gives an exact false-positive rate and recall for that cutoff, with no threshold committed to in advance. Since a transaction is either fraud or non-fraud, exactly one of (false-positive count, true-positive count) changes at each step, so this single pass touches every distinct achievable operating point exactly once.

In [3]:
roc_auc = 0
pr_auc = 0
for i in range(1, len(metrics), 1):
    roc_auc += 0.5 * (metrics[i][1] - metrics[i - 1][1]) * (metrics[i][2] + metrics[i - 1][2])
    pr_auc += (metrics[i][2] - metrics[i - 1][2]) * metrics[i][3]

print(f"ROC-AUC: {roc_auc}, PR-AUC:{pr_auc}")

ROC-AUC: 0.9594099492532535, PR-AUC:0.5643051923340909


**Finding:** ROC-AUC ≈ 0.959 The area under the false-positive-rate/recall curve, computed using the trapezoid sum, indicates that a randomly chosen fraud transaction scores higher than a randomly chosen non-fraud transaction roughly 96% of the time, independent of any specific threshold. PR-AUC ≈ 0.564, computed as a precision-weighted sum across recall steps (average precision), is more important here: with a fraud rate of roughly 0.17%, a detector thats guessing could score close to 0.0017 on this metric by chance alone, so 0.564 represents roughly a 300x improvement over an uninformed baseline.

In [4]:
print(f"Reference ROC-AUC:{roc_auc_score(transactions[:, 1], transactions[:, 2])}")
print(f"Reference PR-AUC: {average_precision_score(transactions[:, 1], transactions[:, 2])}")


Reference ROC-AUC:0.9594099492540198
Reference PR-AUC: 0.5644335268374251


**Finding:** both values match scikit-learn's independent implementations (`roc_auc_score`, `average_precision_score`) to within numerical noise, validating the hand-rolled calculations above the same way the core scoring function was validated against independent references earlier in the project. scikit-learn is used here strictly as an external check on these summary statistics, not as part of the detector itself, which remains fully hand-implemented.

### Selecting an operating threshold

The decision decision between fraud or regular is made by fixing a tolerance for false positives and comparing the recall achieved at that point. The rationale is that since there are so few cases of fraud, it is inevitable to have some false positives; what is important is to flag most of the true positive cases, while avoiding too many false positives. Rather than committing to one tolerance upfront, several candidate false-positive rates (0.1% to 5%) are evaluated side by side, to see the shape of the tradeoff rather than a single, potentially arbitrary, point on it. For each target, the sweep above is searched for the operating point with the largest false-positive rate not exceeding that target.

In [5]:
thresholds = []
for fpr in [0.001, 0.005, 0.01, 0.02, 0.05]:
    fprs = list(np.array(metrics)[:, 1])
    largest_threshold_index = bisect.bisect_right(fprs, fpr)
    thresholds.append(metrics[largest_threshold_index - 1])
print(np.array(thresholds))

[[3.63051999e+01 9.96541415e-04 6.28378378e-01 5.22471910e-01]
 [1.90774252e+01 4.99443109e-03 8.17567568e-01 2.21206581e-01]
 [1.45011536e+01 9.98886218e-03 8.24324324e-01 1.25256674e-01]
 [1.08344918e+01 1.99894484e-02 8.31081081e-01 6.72866521e-02]
 [7.62199024e+00 4.99912070e-02 8.71621622e-01 2.93648987e-02]]


**Finding:** Loosening the tolerance from 0.1% to 0.5% increases recall sharply, from 62.8% to 81.8%; loosening it further, from 0.5% all the way to 5% buys only another ~5% of recall. This makes intuitive sense: once the clearly-separated fraud cases have been captured, the remaining harder-to-separate fraud transactions sit inside the same score range as ordinary transactions, and flagging enough of that range to catch a handful more of them means flagging a large amount of it. This makes a 0.5% false-positive rate a natural cutoff: it captures nearly all of the readily-available recall before the trade-off sharply worsens.

At this operating point, the detector achieves 81.8% recall at a 0.499% false-positive rate, with a threshold of approximately 19.08 and precision of 22.1%. That precision figure is low but also reasonable. With fraud representing  ~0.17% of transactions, false positives are drawn from a pool over 500 times larger than the pool of actual fraud, so even a low false-positive *rate* translates into a large false-positive *count* relative to the number of true positives found. 

This threshold was chosen and evaluated using the same streaming set, rather than a separate split reserved purely for threshold tuning; as a result, the reported recall/precision figures are optimistic relative to how the detector would perform on genuinely unseen data, and should be read as illustrative rather than as a rigorously out-of-sample estimate.